In [1]:

packages = c("reticulate", "tidyverse", "stringr", "kableExtra")

for (pkg in packages) {
    library(pkg, character.only = TRUE, warn.conflicts = FALSE, quietly = TRUE, verbose = FALSE)
}
options(dplyr.width = Inf, dplyr.print_max = 1e9)
options(stringsAsFactors = FALSE)

#### script parameters ####

EMPTY_TEX_STRING = "---"

# ex_df = sudoku_df[
#     ((sudoku_df['concept_noise'] == 0.15) | (sudoku_df['concept_noise'] == 0.0)) 
#     & (sudoku_df['split'] == 'test') 
#     & ((sudoku_df['concept_missing'] == 0.3) | (sudoku_df['concept_missing'] == 0.0))
#     & (sudoku_df['tau'] == 0.05)
#     ]
# ex_df.loc[:, 'has_concept_noise'] = ex_df['concept_noise'] > 0.0


Warning message:
"package 'reticulate' was built under R version 4.3.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.4
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   3.4.4     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.0
v purrr     1.0.2     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [ ]:
sudoku_results <- "../../../results/big_demo/conceptual_safeguards_sudoku.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.05
SPLIT = "test"

sudoku_df <- read_csv(sudoku_results, show_col_types = FALSE)
sudoku_df <- sudoku_df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0)

robot_cs_results <- "../../../results/big_demo/conceptual_safeguards_robot_updated.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

robot_df <- read_csv(robot_cs_results, show_col_types = FALSE)
robot_df <- robot_df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == round(1.05 - target_accuracy_value, 2))) %>%
    mutate(has_concept_noise = concept_noise > 0.0)

In [ ]:
data_order = c("sudoku", "robot")
missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")
difficulty_order = c("easy", "medium", "hard")

df <- rbind(sudoku_df, robot_df)
df$data_name <- factor(df$data_name, levels = data_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$metric <- factor(df$metric, levels = metric_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)
df <- df %>%
    select(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df = table_stats_df %>%
    arrange(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = c(metric, c(has_concept_noise)),
        values_from = svalue,
        names_vary = "slowest"
    )


In [ ]:
cells_df

data_name,target_accuracy_label,concept_missing_mech,coverage_before_FALSE,coverage_after_FALSE,selective_acc_before_FALSE,selective_acc_after_FALSE,coverage_before_TRUE,coverage_after_TRUE,selective_acc_before_TRUE,selective_acc_after_TRUE
<fct>,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
sudoku,easy,none,0.916,0.977,1.000,1.000,0.478,0.988,0.979,0.990
sudoku,easy,mcar,0.892,0.984,0.994,0.996,0.603,0.987,0.761,0.858
sudoku,easy,mnar,0.490,0.981,0.980,0.990,0.657,0.990,0.708,0.807
robot,easy,none,1.000,1.000,1.000,1.000,0.192,0.370,0.859,0.927
robot,easy,mcar,1.000,1.000,1.000,1.000,0.024,0.260,0.636,0.971
robot,easy,mnar,1.000,1.000,1.000,1.000,0.026,0.260,0.667,0.967
robot,medium,none,0.267,0.412,0.732,0.741,0.250,0.414,0.687,0.743
robot,medium,mcar,0.267,0.412,0.732,0.741,0.238,0.407,0.685,0.744
robot,medium,mnar,0.267,0.412,0.732,0.741,0.242,0.410,0.682,0.741


In [ ]:
unite_and_format <- function(data, prefix, new_col_name) {
  data %>%
    unite(!!sym(new_col_name),
      sep = "\\\\",
      c(paste0(prefix, "_FALSE"), paste0(prefix, "_TRUE"))
    ) %>%
    mutate(!!sym(new_col_name) := sprintf("\\cell{r}{%s}", !!sym(new_col_name)))
}
table_df <- cells_df %>%
  group_by(data_name, target_accuracy_label, concept_missing_mech) %>%
  unite_and_format("coverage_before", "cov_before") %>%
  unite_and_format("coverage_after", "cov_after") %>%
  unite_and_format("selective_acc_before", "acc_before") %>%
  unite_and_format("selective_acc_after", "acc_after") %>%
  ungroup()

In [ ]:
table_df

data_name,target_accuracy_label,concept_missing_mech,cov_before,cov_after,acc_before,acc_after
<fct>,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>
sudoku,easy,none,\cell{r}{0.916\\0.478},\cell{r}{0.977\\0.988},\cell{r}{1.000\\0.979},\cell{r}{1.000\\0.990}
sudoku,easy,mcar,\cell{r}{0.892\\0.603},\cell{r}{0.984\\0.987},\cell{r}{0.994\\0.761},\cell{r}{0.996\\0.858}
sudoku,easy,mnar,\cell{r}{0.490\\0.657},\cell{r}{0.981\\0.990},\cell{r}{0.980\\0.708},\cell{r}{0.990\\0.807}
robot,easy,none,\cell{r}{1.000\\0.192},\cell{r}{1.000\\0.370},\cell{r}{1.000\\0.859},\cell{r}{1.000\\0.927}
robot,easy,mcar,\cell{r}{1.000\\0.024},\cell{r}{1.000\\0.260},\cell{r}{1.000\\0.636},\cell{r}{1.000\\0.971}
robot,easy,mnar,\cell{r}{1.000\\0.026},\cell{r}{1.000\\0.260},\cell{r}{1.000\\0.667},\cell{r}{1.000\\0.967}
robot,medium,none,\cell{r}{0.267\\0.250},\cell{r}{0.412\\0.414},\cell{r}{0.732\\0.687},\cell{r}{0.741\\0.743}
robot,medium,mcar,\cell{r}{0.267\\0.238},\cell{r}{0.412\\0.407},\cell{r}{0.732\\0.685},\cell{r}{0.741\\0.744}
robot,medium,mnar,\cell{r}{0.267\\0.242},\cell{r}{0.412\\0.410},\cell{r}{0.732\\0.682},\cell{r}{0.741\\0.741}


In [ ]:
kable_df

data_name,target_accuracy_label,concept_noise,cov_before_none,cov_after_none,acc_before_none,acc_after_none,cov_before_mcar,cov_after_mcar,acc_before_mcar,acc_after_mcar,cov_before_mnar,cov_after_mnar,acc_before_mnar,acc_after_mnar
<fct>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
sudoku,easy,\conceptNoise{},\cell{r}{0.916\\0.478},\cell{r}{0.977\\0.988},\cell{r}{1.000\\0.979},\cell{r}{1.000\\0.990},\cell{r}{0.892\\0.603},\cell{r}{0.984\\0.987},\cell{r}{0.994\\0.761},\cell{r}{0.996\\0.858},\cell{r}{0.490\\0.657},\cell{r}{0.981\\0.990},\cell{r}{0.980\\0.708},\cell{r}{0.990\\0.807}
robot,easy,\conceptNoise{},\cell{r}{1.000\\0.192},\cell{r}{1.000\\0.370},\cell{r}{1.000\\0.859},\cell{r}{1.000\\0.927},\cell{r}{1.000\\0.024},\cell{r}{1.000\\0.260},\cell{r}{1.000\\0.636},\cell{r}{1.000\\0.971},\cell{r}{1.000\\0.026},\cell{r}{1.000\\0.260},\cell{r}{1.000\\0.667},\cell{r}{1.000\\0.967}
robot,medium,\conceptNoise{},\cell{r}{0.267\\0.250},\cell{r}{0.412\\0.414},\cell{r}{0.732\\0.687},\cell{r}{0.741\\0.743},\cell{r}{0.267\\0.238},\cell{r}{0.412\\0.407},\cell{r}{0.732\\0.685},\cell{r}{0.741\\0.744},\cell{r}{0.267\\0.242},\cell{r}{0.412\\0.410},\cell{r}{0.732\\0.682},\cell{r}{0.741\\0.741}
robot,hard,\conceptNoise{},\cell{r}{0.636\\0.572},\cell{r}{0.836\\0.757},\cell{r}{0.512\\0.567},\cell{r}{0.544\\0.580},\cell{r}{0.632\\0.520},\cell{r}{0.835\\0.731},\cell{r}{0.512\\0.566},\cell{r}{0.545\\0.584},\cell{r}{0.631\\0.531},\cell{r}{0.835\\0.740},\cell{r}{0.511\\0.566},\cell{r}{0.545\\0.581}


In [ ]:
kable_df <- table_df %>%
    mutate(
        concept_noise = "\\conceptNoise{}"
    ) %>%
    pivot_wider(
        names_from = c("concept_missing_mech"),
        values_from = c("cov_before", "cov_after", "acc_before", "acc_after"),
        names_vary = "slowest"
    ) %>%
    mutate(data_name = recode(data_name, !!!DATASET_TITLES_MAIN))

top_headers <- c(" " = 3, "none" = 4, "MCAR" = 4, "MNAR" = 4)
mid_headers <- c(" " = 3, c("Cov." = 2, "S.A." = 2) %>% rep(3))
bot_headers <- c("Dataset", "Difficulty", "Concept Noise", c("Before", "After", "Before", "After") %>% rep(3))

overview_table <- kable_df %>%
        kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = bot_headers,
            format = "latex",
            table.envir = NULL,
            linesep = ""
        ) %>%
        add_header_above(mid_headers, bold = FALSE, escape = FALSE) %>%
        add_header_above(top_headers, bold = FALSE, escape = FALSE) %>%
        row_spec(2:nrow(kable_df)-1, hline_after = TRUE, extra_latex_after = "\n")

In [ ]:
overview_table


\begin{tabular}{lllllllllllllll}
\toprule
\multicolumn{3}{c}{ } & \multicolumn{4}{c}{none} & \multicolumn{4}{c}{MCAR} & \multicolumn{4}{c}{MNAR} \\
\cmidrule(l{3pt}r{3pt}){4-7} \cmidrule(l{3pt}r{3pt}){8-11} \cmidrule(l{3pt}r{3pt}){12-15}
\multicolumn{3}{c}{ } & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} \\
\cmidrule(l{3pt}r{3pt}){4-5} \cmidrule(l{3pt}r{3pt}){6-7} \cmidrule(l{3pt}r{3pt}){8-9} \cmidrule(l{3pt}r{3pt}){10-11} \cmidrule(l{3pt}r{3pt}){12-13} \cmidrule(l{3pt}r{3pt}){14-15}
Dataset & Difficulty & Concept Noise & Before & After & Before & After & Before & After & Before & After & Before & After & Before & After\\
\midrule
\sudokuinfo{} & easy & \conceptNoise{} & \cell{r}{0.916\\0.478} & \cell{r}{0.977\\0.988} & \cell{r}{1.000\\0.979} & \cell{r}{1.000\\0.990} & \cell{r}{0.892\\0.603} & \cell{r}{0.984\\0.987} & \cell{r}{0.994\\0.761} & \cell{r}{0.996\\0.858} & \ce

In [ ]:
cells_df = table_stats_df %>%
    arrange(data_name, target_accuracy_label, has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    mutate(metric = recode(metric, !!!c("selective_acc_before" = "sa_before", "selective_acc_after" = "sa_after", "coverage_before" = "cov_before", "coverage_after" = "cov_after"))) %>%
    pivot_wider(
        # names_from = c(metric, c(target_accuracy_label, has_concept_noise)),
        names_from = c(metric),
        values_from = svalue,
        names_vary = "slowest"
    )

In [ ]:
cells_df

data_name,target_accuracy_label,has_concept_noise,concept_missing_mech,cov_before,cov_after,sa_before,sa_after
<fct>,<fct>,<lgl>,<fct>,<chr>,<chr>,<chr>,<chr>
sudoku,easy,FALSE,none,0.916,0.977,1.000,1.000
sudoku,easy,FALSE,mcar,0.892,0.984,0.994,0.996
sudoku,easy,FALSE,mnar,0.490,0.981,0.980,0.990
sudoku,easy,TRUE,none,0.478,0.988,0.979,0.990
sudoku,easy,TRUE,mcar,0.603,0.987,0.761,0.858
sudoku,easy,TRUE,mnar,0.657,0.990,0.708,0.807
robot,easy,FALSE,none,1.000,1.000,1.000,1.000
robot,easy,FALSE,mcar,1.000,1.000,1.000,1.000
robot,easy,FALSE,mnar,1.000,1.000,1.000,1.000


In [ ]:
table_df <- cells_df %>%
  # 1. Pivot to a long format
  pivot_longer(
    cols = starts_with(c("cov", "sa")),
    names_to = c("metric", "level", "boolean_value"),
    names_sep = "_",
    values_to = "value"
  ) %>%
  # 2. Re-unite the desired columns based on common metrics and levels
  group_by(data_name, metric, level, concept_missing_mech) %>%
  summarise(
    combined_value = paste(value, collapse = "\\\\")
  ) %>%
  ungroup() %>%
  # 3. Pivot back to the wide format
  pivot_wider(
    names_from = c(metric, level),
    values_from = combined_value,
    names_sep = "_",
    names_glue = "{metric}_{level}"
  ) %>%
  # 4. Final formatting (optional)
  mutate(across(starts_with(c("cov", "sa")), ~ sprintf("\\cell{r}{%s}", .)))

Warning message:
"Expected 3 pieces. Missing pieces filled with `NA` in 4 rows [1, 2, 3, 4]."


`summarise()` has grouped output by 'data_name', 'metric', 'level'. You can
override using the `.groups` argument.


In [ ]:
table_df

data_name,concept_missing_mech,cov_after,cov_before,sa_after,sa_before
<fct>,<fct>,<chr>,<chr>,<chr>,<chr>
sudoku,none,\cell{r}{0.977\\0.988},\cell{r}{0.916\\0.478},\cell{r}{1.000\\0.990},\cell{r}{1.000\\0.979}
sudoku,mcar,\cell{r}{0.984\\0.987},\cell{r}{0.892\\0.603},\cell{r}{0.996\\0.858},\cell{r}{0.994\\0.761}
sudoku,mnar,\cell{r}{0.981\\0.990},\cell{r}{0.490\\0.657},\cell{r}{0.990\\0.807},\cell{r}{0.980\\0.708}
robot,none,\cell{r}{1.000\\0.370\\0.412\\0.414\\0.836\\0.757},\cell{r}{1.000\\0.192\\0.267\\0.250\\0.636\\0.572},\cell{r}{1.000\\0.927\\0.741\\0.743\\0.544\\0.580},\cell{r}{1.000\\0.859\\0.732\\0.687\\0.512\\0.567}
robot,mcar,\cell{r}{1.000\\0.260\\0.412\\0.407\\0.835\\0.731},\cell{r}{1.000\\0.024\\0.267\\0.238\\0.632\\0.520},\cell{r}{1.000\\0.971\\0.741\\0.744\\0.545\\0.584},\cell{r}{1.000\\0.636\\0.732\\0.685\\0.512\\0.566}
robot,mnar,\cell{r}{1.000\\0.260\\0.412\\0.410\\0.835\\0.740},\cell{r}{1.000\\0.026\\0.267\\0.242\\0.631\\0.531},\cell{r}{1.000\\0.967\\0.741\\0.741\\0.545\\0.581},\cell{r}{1.000\\0.667\\0.732\\0.682\\0.511\\0.566}


In [ ]:
DATASET_TITLES_MAIN <- c(
    "sudoku" = "\\sudokuinfo{}",
    "robot" = "\\robotinfo{}"
)

kable_df <- table_df %>%
    pivot_wider(
        names_from = c("concept_missing_mech"),
        values_from = c("cov_before", "cov_after", "sa_before", "sa_after"),
        names_vary = "slowest"
    ) %>%
    mutate(difficulty = c("\\SudokuDiff{}", "\\RobotDiff{}")) %>%
    relocate(difficulty, .after = data_name) %>%
    mutate(concept_noise = c("\\SudokuConceptNoise{}", "\\RobotConceptNoise{}")) %>%
    relocate(concept_noise, .after = difficulty) %>%
    mutate(data_name = recode(data_name, !!!DATASET_TITLES_MAIN))

top_headers <- c(" " = 3, "\\\\noMissing{}" = 4, "\\\\mcar{}" = 4, "\\\\mnar{}" = 4)
mid_headers <- c(" " = 3, c("Cov." = 2, "S.A." = 2) %>% rep(3))
bot_headers <- c("Dataset", "Difficulty", "Concept Noise", c("Before", "After", "Before", "After") %>% rep(3))

overview_table <- kable_df %>%
        kable(
            booktabs = TRUE,
            escape = FALSE,
            col.names = bot_headers,
            format = "latex",
            table.envir = NULL,
            linesep = ""
        ) %>%
        add_header_above(mid_headers, bold = FALSE, escape = FALSE) %>%
        add_header_above(top_headers, bold = FALSE, escape = FALSE) %>%
        row_spec(2:nrow(kable_df)-1, hline_after = TRUE, extra_latex_after = "\n")

In [ ]:
overview_table


\begin{tabular}{lllllllllllllll}
\toprule
\multicolumn{3}{c}{ } & \multicolumn{4}{c}{\noMissing{}} & \multicolumn{4}{c}{\mcar{}} & \multicolumn{4}{c}{\mnar{}} \\
\cmidrule(l{3pt}r{3pt}){4-7} \cmidrule(l{3pt}r{3pt}){8-11} \cmidrule(l{3pt}r{3pt}){12-15}
\multicolumn{3}{c}{ } & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} & \multicolumn{2}{c}{Cov.} & \multicolumn{2}{c}{S.A.} \\
\cmidrule(l{3pt}r{3pt}){4-5} \cmidrule(l{3pt}r{3pt}){6-7} \cmidrule(l{3pt}r{3pt}){8-9} \cmidrule(l{3pt}r{3pt}){10-11} \cmidrule(l{3pt}r{3pt}){12-13} \cmidrule(l{3pt}r{3pt}){14-15}
Dataset & Difficulty & Concept Noise & Before & After & Before & After & Before & After & Before & After & Before & After & Before & After\\
\midrule
\sudokuinfo{} & \SudokuDiff{} & \SudokuConceptNoise{} & \cell{r}{0.916\\0.478} & \cell{r}{0.977\\0.988} & \cell{r}{1.000\\0.979} & \cell{r}{1.000\\0.990} & \cell{r}{0.892\\0.603} & \cell{r}{0.984\\0.987} & \cell{r}{0.994\\0.761} &

In [ ]:
sudoku_results <- "../../../results/big_demo/conceptual_safeguards_sudoku.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.05
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")

df <- read_csv(sudoku_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

top_headers <- c(" ", " ", "Coverage", "Coverage", "Selective Acc.", "Selective Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Before", "After", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 2, "Coverage" = 2, "Selective Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

\begin{table}[!h]
\centering
\caption{Conceptual Safeguards on Sudoku Dataset}
\centering
\resizebox{\ifdim\width>\linewidth\linewidth\else\width\fi}{!}{
\fontsize{10}{12}\selectfont
\begin{tabular}[t]{cccccc}
\toprule
\multicolumn{2}{c}{ } & \multicolumn{2}{c}{Coverage} & \multicolumn{2}{c}{Selective Acc.} \\
\cmidrule(l{3pt}r{3pt}){3-4} \cmidrule(l{3pt}r{3pt}){5-6}
\textbf{Concept Noise} & \textbf{Concept Missing} & \textbf{Before} & \textbf{After} & \textbf{Before} & \textbf{After}\\
\midrule
FALSE & none & 0.916 & 0.977 & 1.000 & 1.000\\
FALSE & mcar & 0.892 & 0.984 & 0.994 & 0.996\\
FALSE & mnar & 0.490 & 0.981 & 0.980 & 0.990\\
TRUE & none & 0.478 & 0.988 & 0.979 & 0.990\\
TRUE & mcar & 0.603 & 0.987 & 0.761 & 0.858\\
TRUE & mnar & 0.657 & 0.990 & 0.708 & 0.807\\
\bottomrule
\end{tabular}}
\end{table}

In [ ]:
robot_cs_results <- "../../../results/big_demo/conceptual_safeguards_robot.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("coverage_before", "coverage_after", "selective_acc_before", "selective_acc_after")
difficulty_order = c("easy", "medium", "hard")

df <- read_csv(robot_cs_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)

df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0) &
           (tau == TAU)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, target_accuracy_label, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, target_accuracy_label, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

top_headers <- c(" ", " ", "", "Coverage", "Coverage", "Selective Acc.", "Selective Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Before", "After", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 3, "Coverage" = 2, "Selective Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

\begin{table}[!h]
\centering
\caption{Conceptual Safeguards on Sudoku Dataset}
\centering
\resizebox{\ifdim\width>\linewidth\linewidth\else\width\fi}{!}{
\fontsize{10}{12}\selectfont
\begin{tabular}[t]{ccccccc}
\toprule
\multicolumn{3}{c}{ } & \multicolumn{2}{c}{Coverage} & \multicolumn{2}{c}{Selective Acc.} \\
\cmidrule(l{3pt}r{3pt}){4-5} \cmidrule(l{3pt}r{3pt}){6-7}
\textbf{Concept Noise} & \textbf{Concept Missing} & \textbf{Difficulty} & \textbf{Before} & \textbf{After} & \textbf{Before} & \textbf{After}\\
\midrule
FALSE & none & easy & 1.000 & 1.000 & 1.000 & 1.000\\
FALSE & mcar & easy & 1.000 & 1.000 & 1.000 & 1.000\\
FALSE & mnar & easy & 1.000 & 1.000 & 1.000 & 1.000\\
FALSE & none & medium & 0.235 & 0.235 & 0.778 & 0.778\\
FALSE & mcar & medium & 0.235 & 0.235 & 0.778 & 0.778\\
FALSE & mnar & medium & 0.235 & 0.235 & 0.778 & 0.778\\
FALSE & none & hard & 0.000 & 0.000 & NaN & -Inf\\
FALSE & mcar & hard & 0.000 & 0.000 & NaN & -Inf\\
FALSE & mnar & hard & 0.000 & 0.000 & NaN & 

In [ ]:
robot_results <- "../../../results/big_demo/score_intervention_robot.csv"

CONCEPT_NOISE = 0.15
CONCEPT_MISSING = 0.3
TAU = 0.2
SPLIT = "test"

missing_order = c("none", "mcar", "mnar")
metric_order = c("intervened", "overall_acc_before", "overall_acc_after", "acc_non_intervened_before")
difficulty_order = c("easy", "medium", "hard")

df <- read_csv(robot_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$concept_missing_mech <- factor(df$concept_missing_mech, levels = missing_order)
df$target_accuracy_label <- factor(df$target_accuracy_label, levels = difficulty_order)
df <- df %>%
    filter((concept_noise == CONCEPT_NOISE | concept_noise == 0.0) &
           (split == SPLIT) &
           (concept_missing == CONCEPT_MISSING | concept_missing == 0.0)) %>%
    mutate(has_concept_noise = concept_noise > 0.0) %>%
    select(has_concept_noise, concept_missing_mech, target_accuracy_label, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value)) %>%
    mutate(svalue = svalue_dec) %>%
    select(-svalue_pct, -svalue_dec)

cells_df <- table_stats_df %>%
    arrange(has_concept_noise, target_accuracy_label, concept_missing_mech, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    ) %>%
    select(-acc_non_intervened_before)

top_headers <- c(" ", " ", "", "", "Overall Acc.", "Overall Acc.")
mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Intervened", "Before", "After")

table <- cells_df %>%
    kbl(escape = FALSE, align = "c", booktabs = TRUE, linesep = "", 
        col.names = mid_headers, caption = "Conceptual Safeguards on Sudoku Dataset", format="latex") %>%
    add_header_above(header = c(" " = 4, "Overall Acc." = 2)) %>%
    kable_styling(latex_options = c("hold_position", "scale_down"), font_size = 10) %>%
    row_spec(0, bold = TRUE)

table

\begin{table}[!h]
\centering
\caption{Conceptual Safeguards on Sudoku Dataset}
\centering
\resizebox{\ifdim\width>\linewidth\linewidth\else\width\fi}{!}{
\fontsize{10}{12}\selectfont
\begin{tabular}[t]{cccccc}
\toprule
\multicolumn{4}{c}{ } & \multicolumn{2}{c}{Overall Acc.} \\
\cmidrule(l{3pt}r{3pt}){5-6}
\textbf{Concept Noise} & \textbf{Concept Missing} & \textbf{Difficulty} & \textbf{Intervened} & \textbf{Before} & \textbf{After}\\
\midrule
FALSE & none & easy & 0.762 & 1.000 & 1.000\\
FALSE & mcar & easy & 0.762 & 1.000 & 1.000\\
FALSE & mnar & easy & 0.762 & 1.000 & 1.000\\
FALSE & none & medium & 1.000 & 0.784 & 0.784\\
FALSE & mcar & medium & 1.000 & 0.784 & 0.784\\
FALSE & mnar & medium & 1.000 & 0.784 & 0.784\\
FALSE & none & hard & 0.000 & 0.593 & 0.593\\
FALSE & mcar & hard & 0.000 & 0.593 & 0.593\\
FALSE & mnar & hard & 0.000 & 0.593 & 0.593\\
TRUE & none & easy & 0.761 & 1.000 & 1.000\\
TRUE & mcar & easy & 0.761 & 1.000 & 1.000\\
TRUE & mnar & easy & 0.761 & 1.000 & 1.000

In [63]:
robot_results <- "../../../results/robot_demo_results.csv"

# CONCEPT_NOISE = 0.15
# CONCEPT_MISSING = 0.3
# SPLIT = "test"

TAU = 0.6
metric_order = c("accuracy", "predictions_intervened_on", "total_concept_edits_made")
model_order = c("dnn", "cbm_no_int", "cbm_with_int_1", "cbm_with_int_3")
DATASET_TITLES <- c(
    "ideal" = "\\robotIdeal{}",
    "subconcept" = "\\robotSubconcept{}"
)

df <- read_csv(robot_results, show_col_types = FALSE)
df$metric <- factor(df$metric, levels = metric_order)
df$model <- factor(df$model, levels = model_order)
df <- df %>%
    # mutate(model = ifelse(is.na(budget), model, paste0(model, "_", budget))) %>%
    filter((threshold == TAU) | is.na(threshold)) %>%
    select(data_name, model, metric, value)

table_stats_df <- df %>%
    mutate(svalue_pct = sprintf("%1.1f\\%%", 100 * value),
        svalue_dec = sprintf("%1.3f", value),
        svalue_int = sprintf("%d", round(value))) %>%
    mutate(svalue = ifelse(metric == "accuracy", svalue_dec,
                        ifelse(metric == "predictions_intervened_on", svalue_int,
                               svalue_int))) %>%
    select(-svalue_pct, -svalue_dec, -svalue_int)

cells_df <- table_stats_df %>%
    arrange(model, metric) %>%
    select(-value) %>%
    pivot_wider(
        names_from = metric,
        values_from = svalue
    )

cells_df[is.na(cells_df)] <- EMPTY_TEX_STRING

table_df <- cells_df %>%
    group_by(data_name, model) %>%
    unite(cell_str, sep = "\\\\", metric_order) %>%
    mutate(cell_str = sprintf("\\cell{r}{%s}\n", cell_str)) %>%
    ungroup() %>%
    arrange(data_name, model)

kable_df <- table_df %>%
    mutate(
        metrics = "\\robotMetrics{}",
        data_name = recode(data_name, !!!DATASET_TITLES)
    ) %>%
    pivot_wider(
        names_from = c(model),
        values_from = cell_str,
        names_sort = FALSE,
        names_glue = "{model}",
    )

kable_df[is.na(kable_df)] <- "\\cell{r}{---\\\\---\\\\---}"

top_headers <- c("Dataset", "Metrics", "DNN", "CBM No Int.", "CBM With Int. (1)", "CBM With Int. (3)")
# mid_headers <- c("Concept Noise", "Concept Missing", "Difficulty", "Intervened", "Before", "After")

table <- kable_df %>%
    kbl(escape = FALSE, toprule = '', align = "l", booktabs = TRUE, linesep = "\\midrule", 
        col.names = top_headers, format="latex")